# 🏛️ Phase 3: Internet Archive Wayback Machine PPTX Mining

Downloads PPT/PPTX files from the Internet Archive's Wayback Machine.

**How it works:**
1. Query the CDX API for each university domain
2. Filter by PPT/PPTX MIME types
3. Download archived files via Wayback Machine
4. Apply all standard filters

**Expected yield:** 50K-150K unique files

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
warnings.filterwarnings('ignore')

DRIVE_DIR = '/content/drive/MyDrive/PPTX_Global_Collection'
SEEN_FILE = os.path.join(DRIVE_DIR, 'seen_tags.txt')
IA_PROGRESS = os.path.join(DRIVE_DIR, 'ia_progress.json')
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Output: {DRIVE_DIR}')

In [ ]:
# Cell 2: Config
import hashlib, json, os, re, time, zipfile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

MIN_SIZE = 2 * 1024 * 1024
IA_CDX = 'https://web.archive.org/cdx/search/cdx'
IA_DL  = 'https://web.archive.org/web'

CHINA_KW = ['chinese','china','hong kong','taiwan','beijing','shanghai',
            'mandarin','wuhan','guangzhou','shenzhen','nanjing','zhonghua',
            '.cn/', '.edu.cn', '.ac.cn', '.hk/', '.tw/']

# University domains from all 71 countries
# Each entry: (domain_pattern, country_label)
UNIVERSITY_DOMAINS = [
    # Africa
    ('*.ac.za', 'ZA'), ('*.edu.za', 'ZA'), ('*.edu.ng', 'NG'),
    ('*.edu.eg', 'EG'), ('*.ac.ke', 'KE'), ('*.edu.ke', 'KE'),
    ('*.ac.ma', 'MA'), ('*.edu.gh', 'GH'), ('*.edu.tn', 'TN'),
    # Asia
    ('*.ac.in', 'IN'), ('*.edu.in', 'IN'), ('*.ac.jp', 'JP'),
    ('*.ac.kr', 'KR'), ('*.edu.sg', 'SG'), ('*.edu.my', 'MY'),
    ('*.edu.tr', 'TR'), ('*.ac.id', 'ID'), ('*.edu.ph', 'PH'),
    ('*.edu.pk', 'PK'), ('*.edu.bd', 'BD'), ('*.ac.ae', 'AE'),
    ('*.ac.ir', 'IR'), ('*.edu.sa', 'SA'), ('*.edu.jo', 'JO'),
    ('*.edu.lb', 'LB'), ('*.edu.qa', 'QA'), ('*.ac.lk', 'LK'),
    ('*.edu.kz', 'KZ'), ('*.ac.th', 'TH'), ('*.edu.vn', 'VN'),
    ('*.ac.il', 'IL'),
    # Europe
    ('*.ac.uk', 'UK'), ('*.edu.fr', 'FR'), ('*.uni-*.de', 'DE'),
    ('*.univ-*.fr', 'FR'),
    ('*.edu.it', 'IT'), ('*.edu.es', 'ES'), ('*.edu.nl', 'NL'),
    ('*.edu.ch', 'CH'), ('*.ac.be', 'BE'), ('*.ac.at', 'AT'),
    ('*.edu.se', 'SE'), ('*.edu.no', 'NO'), ('*.edu.dk', 'DK'),
    ('*.edu.fi', 'FI'), ('*.edu.ie', 'IE'), ('*.edu.pt', 'PT'),
    ('*.edu.gr', 'GR'), ('*.edu.pl', 'PL'), ('*.edu.cz', 'CZ'),
    ('*.edu.hu', 'HU'), ('*.edu.ro', 'RO'), ('*.edu.ua', 'UA'),
    ('*.edu.hr', 'HR'), ('*.edu.rs', 'RS'), ('*.edu.ru', 'RU'),
    ('*.ac.ru', 'RU'), ('*.edu.bg', 'BG'), ('*.edu.sk', 'SK'),
    ('*.edu.lt', 'LT'), ('*.edu.si', 'SI'), ('*.edu.ee', 'EE'),
    # Americas
    ('*.edu', 'GLOBAL'), ('*.edu.ca', 'CA'), ('*.edu.mx', 'MX'),
    ('*.edu.br', 'BR'), ('*.edu.ar', 'AR'), ('*.edu.cl', 'CL'),
    ('*.edu.co', 'CO'), ('*.edu.pe', 'PE'), ('*.edu.ec', 'EC'),
    ('*.edu.ve', 'VE'), ('*.edu.cr', 'CR'), ('*.edu.uy', 'UY'),
    ('*.edu.cu', 'CU'),
    # Oceania
    ('*.edu.au', 'AU'), ('*.ac.nz', 'NZ'),
    # High-value specific institutions
    ('*.mit.edu', 'MIT'), ('*.stanford.edu', 'STAN'),
    ('*.cam.ac.uk', 'CAM'), ('*.ox.ac.uk', 'OX'),
    ('*.ethz.ch', 'ETH'), ('*.epfl.ch', 'EPFL'),
]

session = requests.Session()
retry = Retry(total=3, backoff_factor=2, status_forcelist=[429,500,502,503,504])
session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers.update({'User-Agent': 'Mozilla/5.0 Academic-Research-Bot/3.0'})

def load_seen():
    tags = set()
    if os.path.exists(SEEN_FILE):
        with open(SEEN_FILE) as f:
            tags = {l.strip() for l in f if l.strip()}
    return tags

def save_tag(tag):
    with open(SEEN_FILE, 'a') as f: f.write(tag + '\n')

def is_china_url(url):
    u = url.lower()
    return any(kw in u for kw in CHINA_KW)

def has_chinese_chars(fp):
    try:
        with zipfile.ZipFile(fp, 'r') as z:
            for n in z.namelist():
                if 'slide' in n and n.endswith('.xml'):
                    d = z.read(n).decode('utf-8', errors='ignore')
                    if len(re.findall(r'[\u4e00-\u9fff]', d)) > 50: return True
    except: pass
    return False

def valid_ppt(fp):
    try:
        with open(fp, 'rb') as f: h = f.read(8)
        return h[:2] == b'PK' or h[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    except: return False

def load_ia_progress():
    if os.path.exists(IA_PROGRESS):
        with open(IA_PROGRESS) as f: return json.load(f)
    return {'completed_domains': [], 'downloaded': 0}

def save_ia_progress(p):
    with open(IA_PROGRESS, 'w') as f: json.dump(p, f)

print(f'✅ Config ready: {len(UNIVERSITY_DOMAINS)} domain patterns')

In [ ]:
# Cell 3: CDX Query + Download Engine
seen = load_seen()
ia_prog = load_ia_progress()
stats = {'domains_queried': 0, 'urls_found': 0, 'downloaded': 0,
         'small': 0, 'china': 0, 'invalid': 0, 'dup': 0, 'errors': 0}

def query_ia_cdx(domain_pattern):
    """Query Internet Archive CDX for PPT/PPTX files on a domain."""
    all_urls = []
    for mime in ['application/vnd.ms-powerpoint',
                 'application/vnd.openxmlformats-officedocument.presentationml.presentation']:
        params = {
            'url': domain_pattern,
            'matchType': 'domain',
            'filter': f'mimetype:{mime}',
            'collapse': 'urlkey',  # Deduplicate by URL
            'output': 'json',
            'fl': 'original,timestamp,mimetype,length',
            'limit': 10000,
        }
        try:
            r = session.get(IA_CDX, params=params, timeout=120)
            if r.ok and r.text.strip():
                lines = r.json()
                # First line is header
                if lines and isinstance(lines[0], list):
                    header = lines[0]
                    for row in lines[1:]:
                        entry = dict(zip(header, row))
                        all_urls.append(entry)
        except Exception as e:
            pass
        time.sleep(1)  # Rate limit between MIME type queries
    return all_urls

def download_from_ia(entry):
    """Download a file from Internet Archive."""
    global seen, stats
    original_url = entry.get('original', '')
    timestamp = entry.get('timestamp', '')
    if not original_url: return False

    tag = hashlib.sha1(original_url.encode()).hexdigest()[:10]
    if tag in seen: stats['dup'] += 1; return False
    if is_china_url(original_url): stats['china'] += 1; return False

    # Check length if available
    try:
        content_len = int(entry.get('length', 0))
        if 0 < content_len < MIN_SIZE: stats['small'] += 1; return False
    except: pass

    # Construct Wayback URL
    wb_url = f'{IA_DL}/{timestamp}id_/{original_url}'

    url_lower = original_url.lower().split('?')[0]
    ext = '.ppt' if url_lower.endswith('.ppt') else '.pptx'
    safe = re.sub(r'[^\w]', '_', original_url.split('/')[-1][:35])
    fname = f'{tag}_ia_{safe}{ext}'
    dest = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(dest): seen.add(tag); return False

    try:
        r = session.get(wb_url, timeout=90, stream=True)
        if not r.ok: stats['errors'] += 1; return False

        with open(dest, 'wb') as f:
            for chunk in r.iter_content(65536): f.write(chunk)

        sz = os.path.getsize(dest)
        if sz < MIN_SIZE: os.remove(dest); stats['small'] += 1; return False
        if not valid_ppt(dest): os.remove(dest); stats['invalid'] += 1; return False
        if has_chinese_chars(dest): os.remove(dest); stats['china'] += 1; return False

        seen.add(tag); save_tag(tag)
        stats['downloaded'] += 1
        return True
    except:
        if os.path.exists(dest): os.remove(dest)
        stats['errors'] += 1; return False

print('✅ IA engine ready')

In [ ]:
# Cell 4: Run the mining!
TARGET = 150000
completed = set(ia_prog.get('completed_domains', []))

print(f'🚀 Starting Internet Archive mining')
print(f'   Target: {TARGET} files')
print(f'   Domains: {len(UNIVERSITY_DOMAINS)}')
print(f'   Already completed: {len(completed)} domains')
print()

for di, (domain, country) in enumerate(UNIVERSITY_DOMAINS):
    if stats['downloaded'] >= TARGET: break
    if domain in completed: continue

    print(f'\n🌐 [{di+1}/{len(UNIVERSITY_DOMAINS)}] {domain} ({country}) | '
          f'Total: {stats["downloaded"]}/{TARGET}')

    urls = query_ia_cdx(domain)
    stats['domains_queried'] += 1
    stats['urls_found'] += len(urls)

    if urls:
        print(f'   Found {len(urls)} PPT/PPTX URLs')
        for ui, entry in enumerate(urls):
            if stats['downloaded'] >= TARGET: break
            if download_from_ia(entry):
                if stats['downloaded'] % 25 == 0:
                    print(f'   ✅ [{stats["downloaded"]}] Downloaded | '
                          f'Dup: {stats["dup"]} | Small: {stats["small"]}')
            if ui % 20 == 0: time.sleep(1)  # Rate limit
    else:
        print(f'   No PPT files found')

    completed.add(domain)
    ia_prog['completed_domains'] = list(completed)
    ia_prog['downloaded'] = stats['downloaded']
    save_ia_progress(ia_prog)
    time.sleep(2)  # Be nice to Internet Archive

print(f'\n{"="*60}')
print(f'📊 INTERNET ARCHIVE RESULTS')
print(f'{"="*60}')
for k,v in stats.items(): print(f'   {k:20s}: {v}')
print(f'{"="*60}')

with open(os.path.join(DRIVE_DIR, 'ia_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

In [ ]:
# Cell 5: Combined summary
import glob
files = glob.glob(os.path.join(DRIVE_DIR,'*.pptx')) + glob.glob(os.path.join(DRIVE_DIR,'*.ppt'))
total_gb = sum(os.path.getsize(f) for f in files) / (1024**3)

sources = {}
for f in files:
    bn = os.path.basename(f)
    if '_ia_' in bn: sources['Internet Archive'] = sources.get('Internet Archive', 0) + 1
    elif '_cc_' in bn: sources['Common Crawl'] = sources.get('Common Crawl', 0) + 1
    elif '_p4w_' in bn: sources['ppt4web'] = sources.get('ppt4web', 0) + 1
    elif '_pon_' in bn: sources['pptonline'] = sources.get('pptonline', 0) + 1
    elif '_fig_' in bn: sources['Figshare'] = sources.get('Figshare', 0) + 1
    elif '_zen_' in bn: sources['Zenodo'] = sources.get('Zenodo', 0) + 1
    elif '_hal_' in bn: sources['HAL'] = sources.get('HAL', 0) + 1
    else: sources['Other'] = sources.get('Other', 0) + 1

print(f'📊 GLOBAL COLLECTION SUMMARY')
print(f'{'='*40}')
for src, cnt in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'   {src:20s}: {cnt:>6,}')
print(f'{'='*40}')
print(f'   {"TOTAL":20s}: {len(files):>6,} files | {total_gb:.2f} GB')